Construção da camada Gold


In [1]:
import os
import platform
from pathlib import Path

import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, col, count, current_timestamp, lit, round, sum, when

In [2]:
spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('construcao_camada_gold')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('WARN')
print('Versão do Spark:', spark.version)

Versão do Spark: 3.5.9


In [3]:
pasta_atual = Path.cwd()

if pasta_atual.name == 'notebooks':
    raiz_projeto = pasta_atual.parent
else:
    raiz_projeto = pasta_atual

pasta_silver = raiz_projeto / 'data' / 'silver'
pasta_gold = raiz_projeto / 'data' / 'gold'
pasta_gold.mkdir(parents=True, exist_ok=True)

print('Silver:', pasta_silver)
print('Gold:', pasta_gold)

Silver: c:\Users\claud\Documents\Py\tech_challenge_02\data\silver
Gold: c:\Users\claud\Documents\Py\tech_challenge_02\data\gold


Verificação do ambiente


In [4]:
hadoop_home = os.getenv('HADOOP_HOME')
winutils_existe = bool(
    hadoop_home and (Path(hadoop_home) / 'bin' / 'winutils.exe').exists()
)
usar_spark = platform.system() != 'Windows' or winutils_existe

print('Processamento:', 'PySpark' if usar_spark else 'Pandas para teste local')

Processamento: Pandas para teste local


Leitura da camada Silver

In [5]:
nomes_tabelas = [
    'alunos',
    'municipio',
    'uf',
    'meta_alfabetizacao_municipio',
    'meta_alfabetizacao_uf',
    'meta_alfabetizacao_brasil',
]

dados_silver = {}

for nome_tabela in nomes_tabelas:
    caminho_tabela = pasta_silver / nome_tabela
    if usar_spark:
        dados_silver[nome_tabela] = spark.read.parquet(caminho_tabela.as_posix())
    else:
        arquivos = sorted(caminho_tabela.glob('*.parquet'))
        if not arquivos:
            raise FileNotFoundError(f'Tabela Silver não encontrada: {nome_tabela}')
        dados_silver[nome_tabela] = pd.concat(
            [pd.read_parquet(arquivo) for arquivo in arquivos],
            ignore_index=True,
        )
    print('Tabela Silver lida:', nome_tabela)

Tabela Silver lida: alunos
Tabela Silver lida: municipio
Tabela Silver lida: uf
Tabela Silver lida: meta_alfabetizacao_municipio
Tabela Silver lida: meta_alfabetizacao_uf
Tabela Silver lida: meta_alfabetizacao_brasil


Indicadores por município


In [6]:
if usar_spark:
    alunos_municipais = (
        dados_silver['alunos']
        .filter((col('rede') == '3') & (col('presenca') == 1))
        .groupBy('ano', 'id_municipio')
        .agg(
            count('*').alias('alunos_presentes'),
            sum('alfabetizado').alias('alunos_alfabetizados'),
            avg('proficiencia').alias('media_proficiencia'),
        )
        .withColumn(
            'taxa_microdados',
            round(col('alunos_alfabetizados') * 100 / col('alunos_presentes'), 2),
        )
    )

    resultado_municipio = (
        dados_silver['municipio']
        .filter(col('rede') == '3')
        .select(
            'ano', 'id_municipio',
            col('taxa_alfabetizacao').alias('taxa_oficial'),
            'media_portugues',
        )
    )

    metas_municipio = (
        dados_silver['meta_alfabetizacao_municipio']
        .select(
            'ano', 'id_municipio', 'nivel_alfabetizacao',
            'percentual_participacao', 'meta_alfabetizacao_2024',
        )
    )

    gold_municipio = (
        alunos_municipais
        .join(resultado_municipio, ['ano', 'id_municipio'], 'left')
        .join(metas_municipio, ['ano', 'id_municipio'], 'left')
        .withColumn(
            'meta_do_ano',
            when(col('ano') == 2024, col('meta_alfabetizacao_2024')),
        )
        .withColumn('diferenca_para_meta', round(col('taxa_oficial') - col('meta_do_ano'), 2))
        .withColumn(
            'situacao_meta',
            when(col('meta_do_ano').isNull(), 'Sem meta')
            .when(col('taxa_oficial') >= col('meta_do_ano'), 'Atingida')
            .otherwise('Não atingida'),
        )
        .withColumn('_data_criacao_gold', current_timestamp())
    )
else:
    alunos = dados_silver['alunos']
    alunos_municipais = (
        alunos[(alunos['rede'] == '3') & (alunos['presenca'] == 1)]
        .groupby(['ano', 'id_municipio'], as_index=False)
        .agg(
            alunos_presentes=('id_aluno', 'count'),
            alunos_alfabetizados=('alfabetizado', 'sum'),
            media_proficiencia=('proficiencia', 'mean'),
        )
    )
    alunos_municipais['taxa_microdados'] = (
        alunos_municipais['alunos_alfabetizados'] * 100
        / alunos_municipais['alunos_presentes']
    ).round(2)

    resultado_municipio = dados_silver['municipio'].query("rede == '3'")[
        ['ano', 'id_municipio', 'taxa_alfabetizacao', 'media_portugues']
    ].rename(columns={'taxa_alfabetizacao': 'taxa_oficial'})

    metas_municipio = dados_silver['meta_alfabetizacao_municipio'][[
        'ano', 'id_municipio', 'nivel_alfabetizacao',
        'percentual_participacao', 'meta_alfabetizacao_2024',
    ]]

    gold_municipio = alunos_municipais.merge(
        resultado_municipio, on=['ano', 'id_municipio'], how='left'
    ).merge(metas_municipio, on=['ano', 'id_municipio'], how='left')
    gold_municipio['meta_do_ano'] = gold_municipio['meta_alfabetizacao_2024'].where(
        gold_municipio['ano'] == 2024
    )
    gold_municipio['diferenca_para_meta'] = (
        gold_municipio['taxa_oficial'] - gold_municipio['meta_do_ano']
    ).round(2)
    gold_municipio['situacao_meta'] = 'Sem meta'
    tem_meta = gold_municipio['meta_do_ano'].notna()
    gold_municipio.loc[tem_meta, 'situacao_meta'] = 'Não atingida'
    gold_municipio.loc[tem_meta & (gold_municipio['taxa_oficial'] >= gold_municipio['meta_do_ano']), 'situacao_meta'] = 'Atingida'
    gold_municipio['_data_criacao_gold'] = pd.Timestamp.now()

Indicadores por UF

In [7]:
if usar_spark:
    resultado_uf = (
        dados_silver['uf']
        .filter(col('rede') == '5')
        .select(
            'ano', 'sigla_uf',
            col('taxa_alfabetizacao').alias('taxa_oficial'),
            'media_portugues',
        )
    )
    metas_uf = dados_silver['meta_alfabetizacao_uf'].select(
        'ano', 'sigla_uf', 'percentual_participacao', 'meta_alfabetizacao_2024'
    )
    gold_uf = (
        resultado_uf
        .join(metas_uf, ['ano', 'sigla_uf'], 'left')
        .withColumn(
            'meta_do_ano',
            when(col('ano') == 2024, col('meta_alfabetizacao_2024')),
        )
        .withColumn('diferenca_para_meta', round(col('taxa_oficial') - col('meta_do_ano'), 2))
        .withColumn(
            'situacao_meta',
            when(col('meta_do_ano').isNull(), 'Sem meta')
            .when(col('taxa_oficial') >= col('meta_do_ano'), 'Atingida')
            .otherwise('Não atingida'),
        )
        .withColumn('_data_criacao_gold', current_timestamp())
    )
else:
    resultado_uf = dados_silver['uf'].query("rede == '5'")[[
        'ano', 'sigla_uf', 'taxa_alfabetizacao', 'media_portugues'
    ]].rename(columns={'taxa_alfabetizacao': 'taxa_oficial'})
    metas_uf = dados_silver['meta_alfabetizacao_uf'][[
        'ano', 'sigla_uf', 'percentual_participacao', 'meta_alfabetizacao_2024'
    ]]
    gold_uf = resultado_uf.merge(metas_uf, on=['ano', 'sigla_uf'], how='left')
    gold_uf['meta_do_ano'] = gold_uf['meta_alfabetizacao_2024'].where(gold_uf['ano'] == 2024)
    gold_uf['diferenca_para_meta'] = (gold_uf['taxa_oficial'] - gold_uf['meta_do_ano']).round(2)
    gold_uf['situacao_meta'] = 'Sem meta'
    tem_meta = gold_uf['meta_do_ano'].notna()
    gold_uf.loc[tem_meta, 'situacao_meta'] = 'Não atingida'
    gold_uf.loc[tem_meta & (gold_uf['taxa_oficial'] >= gold_uf['meta_do_ano']), 'situacao_meta'] = 'Atingida'
    gold_uf['_data_criacao_gold'] = pd.Timestamp.now()

Indicadores do Brasil

In [8]:
if usar_spark:
    gold_brasil = (
        dados_silver['meta_alfabetizacao_brasil']
        .select(
            'ano',
            col('taxa_alfabetizacao').alias('taxa_oficial'),
            'percentual_participacao', 'meta_alfabetizacao_2024',
        )
        .withColumn(
            'meta_do_ano',
            when(col('ano') == 2024, col('meta_alfabetizacao_2024')),
        )
        .withColumn('diferenca_para_meta', round(col('taxa_oficial') - col('meta_do_ano'), 2))
        .withColumn(
            'situacao_meta',
            when(col('meta_do_ano').isNull(), 'Sem meta')
            .when(col('taxa_oficial') >= col('meta_do_ano'), 'Atingida')
            .otherwise('Não atingida'),
        )
        .withColumn('_data_criacao_gold', current_timestamp())
    )
else:
    gold_brasil = dados_silver['meta_alfabetizacao_brasil'][[
        'ano', 'taxa_alfabetizacao', 'percentual_participacao',
        'meta_alfabetizacao_2024',
    ]].rename(columns={'taxa_alfabetizacao': 'taxa_oficial'}).copy()
    gold_brasil['meta_do_ano'] = gold_brasil['meta_alfabetizacao_2024'].where(
        gold_brasil['ano'] == 2024
    )
    gold_brasil['diferenca_para_meta'] = (
        gold_brasil['taxa_oficial'] - gold_brasil['meta_do_ano']
    ).round(2)
    gold_brasil['situacao_meta'] = 'Sem meta'
    tem_meta = gold_brasil['meta_do_ano'].notna()
    gold_brasil.loc[tem_meta, 'situacao_meta'] = 'Não atingida'
    gold_brasil.loc[tem_meta & (gold_brasil['taxa_oficial'] >= gold_brasil['meta_do_ano']), 'situacao_meta'] = 'Atingida'
    gold_brasil['_data_criacao_gold'] = pd.Timestamp.now()

Gravação da camada Gold

In [9]:
tabelas_gold = {
    'indicadores_municipio': gold_municipio,
    'indicadores_uf': gold_uf,
    'indicadores_brasil': gold_brasil,
}

for nome_tabela, df in tabelas_gold.items():
    caminho_saida = pasta_gold / nome_tabela
    if usar_spark:
        df.write.mode('overwrite').parquet(caminho_saida.as_posix())
    else:
        caminho_saida.mkdir(parents=True, exist_ok=True)
        df.to_parquet(caminho_saida / 'data.parquet', index=False)
    print('Tabela Gold gravada:', nome_tabela)

Tabela Gold gravada: indicadores_municipio
Tabela Gold gravada: indicadores_uf
Tabela Gold gravada: indicadores_brasil


Validação e visualização

In [12]:
resultado_validacao = []

for nome_tabela, df in tabelas_gold.items():
    quantidade = df.count() if usar_spark else len(df)
    resultado_validacao.append({
        'tabela': nome_tabela,
        'registros': quantidade,
    })

display(pd.DataFrame(resultado_validacao))

if usar_spark:
    gold_uf.orderBy('ano', 'sigla_uf').show(10, truncate=False)
else:
    display(gold_uf.sort_values(['ano', 'sigla_uf']).tail(10))

,tabela,registros
0,indicadores_municipio,10272
1,indicadores_uf,49
2,indicadores_brasil,3


,ano,sigla_uf,taxa_oficial,media_portugues,percentual_participacao,meta_alfabetizacao_2024,meta_do_ano,diferenca_para_meta,situacao_meta,_data_criacao_gold
29,2024,PI,59.82,754.81,95.09,57.0,57.0,2.82,Atingida,2026-08-30 20:30:42.154813
28,2024,PR,70.42,754.66,86.25,74.2,74.2,-3.78,Não atingida,2026-08-30 20:30:42.154813
41,2024,RJ,55.25,738.9,83.12,56.7,56.7,-1.45,Não atingida,2026-08-30 20:30:42.154813
46,2024,RN,39.29,730.44,77.72,43.8,43.8,-4.51,Não atingida,2026-08-30 20:30:42.154813
31,2024,RO,62.62,745.52,88.33,67.1,67.1,-4.48,Não atingida,2026-08-30 20:30:42.154813
44,2024,RS,44.67,734.2547,82.86,66.2,66.2,-21.53,Não atingida,2026-08-30 20:30:42.154813
40,2024,SC,62.02,750.1,70.05,64.5,64.5,-2.48,Não atingida,2026-08-30 20:30:42.154813
48,2024,SE,38.39,723.56,92.84,38.3,38.3,0.09,Atingida,2026-08-30 20:30:42.154813
37,2024,SP,58.13,749.8962,89.12,56.6,56.6,1.53,Atingida,2026-08-30 20:30:42.154813
33,2024,TO,50.07,742.86,85.15,49.5,49.5,0.57,Atingida,2026-08-30 20:30:42.154813


In [13]:
spark.stop()